### Setup

In [1]:
# Library
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
os.environ["CUDA_VISIBLE_DEVICES"]="1"
import torch
from torchvision import transforms
from diffusers import StableDiffusionPipeline
from PIL import Image
from torchmetrics.image.fid import FrechetInceptionDistance

# GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device : ',device)   

# CONFIG
TEST_SIZE = 100

# PATH
## Augment Instruction Performace Data
SEED_IMAGE_FOLDER = '../Data/Seed/Seed_Image'
SEED_IMAGE_FOLDER = sorted(os.listdir(SEED_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
SEED_LABEL_FOLDER ='../Data/Seed/EN_Seed_Label'
SDDE_LABEL_FILE = sorted(os.listdir(SEED_LABEL_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
AUGEMNT_IMAGE_FOLDER = '../Data/Augment/Augment_KFashion_Image'
AUGMENT_IMAGE_FILE = sorted(os.listdir(AUGEMNT_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
AUGEMNT_LABEL_FOLDER =  '../Data/Augment/Augment_CLIP'
AUGMENT_LABEL_FILE = sorted(os.listdir(AUGEMNT_LABEL_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
## Generate of Model Performance-Prompt Following Performance
PROMPT_FOLDER = '../Data/Generate/Prompt'
PROMPT_FILE = sorted(os.listdir(PROMPT_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
SD_IMAGE_FOLDER = '../Data/Generate/Generate_Image/StableDiffusion_Image'
SD_IMAGE_FILE = sorted(os.listdir(SD_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
FIGMA_IMAGE_FOLDER = '../Data/Generate/Generate_Image/FIGMA_Image'
FIGMA_IMAGE_FILE = sorted(os.listdir(FIGMA_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
## Generate of Model Performance-Image Following Performance
ANSWER_IMAGE_FOLDER = '../Data/Generate/Answer_Image'
ANSWER_IMAGE_FILE = sorted(os.listdir(ANSWER_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
SD_IMAGE_FOLDER = '../Data/Generate/Generate_Image/StableDiffusion_Image'
SD_IMAGE_FILE = sorted(os.listdir(SD_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
FIGMA_IMAGE_FOLDER = '../Data/Generate/Generate_Image/FIGMA_Image'
FIGMA_IMAGE_FILE = sorted(os.listdir(FIGMA_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
## MODEL
PRE_TRAINED_MODEL_NAME="stablediffusionapi/deliberate-v2"
SAVE_WEIGHTS_PATH = '../Experiment/model_weights/FIGMA_weights_20250228_131700'

2025-02-28 15:18:09.716618: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-02-28 15:18:09.747638: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-02-28 15:18:10.304639: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


device :  cuda


### Generate Image by Stable Diffusion

In [2]:
# 모델 로드
satblediffusion_pipe = StableDiffusionPipeline.from_pretrained(PRE_TRAINED_MODEL_NAME, torch_dtype=torch.float16)
satblediffusion_pipe.to("cuda");

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

/home/gayeon38/anaconda3/envs/vlm-env/lib/python3.8/site-packages/transformers/models/clip/feature_extraction_clip.py:28: FutureWarning: The class CLIPFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use CLIPImageProcessor instead.
  warnings.warn(


In [ ]:
for idx in range(len(PROMPT_FILE)):
    prompt_path = os.path.join(PROMPT_FOLDER, PROMPT_FILE[idx])
    with open(prompt_path, 'r') as f:
        prompt = f.read()[16:]
    generated_image = satblediffusion_pipe(prompt).images[0]
    save_image_path = SD_IMAGE_FOLDER + "/" + PROMPT_FILE[idx][:-5] + ".jpg"
    generated_image.save(save_image_path)
    print(f"Image is Saved in {save_image_path}")

### Generate Image by FIGMA

In [2]:
# 학습된 모델 로드
figma_pipe = StableDiffusionPipeline.from_pretrained(PRE_TRAINED_MODEL_NAME, torch_dtype=torch.float16)
figma_pipe.to("cuda");
figma_pipe.load_lora_weights(SAVE_WEIGHTS_PATH, safe_serialization=True)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

/home/gayeon38/anaconda3/envs/vlm-env/lib/python3.8/site-packages/transformers/models/clip/feature_extraction_clip.py:28: FutureWarning: The class CLIPFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use CLIPImageProcessor instead.
  warnings.warn(


In [ ]:
for idx in range(len(PROMPT_FILE)):
    prompt_path = os.path.join(PROMPT_FOLDER, PROMPT_FILE[idx])
    with open(prompt_path, 'r') as f:
        prompt = f.read()[16:]
    generated_image = figma_pipe(prompt).images[0]
    save_image_path = FIGMA_IMAGE_FOLDER + "/" + PROMPT_FILE[idx][:-5] + ".jpg"
    generated_image.save(save_image_path)
    print(f"Image is Saved in {save_image_path}")